## Plotting stacked absorption profiles compared to Lyman-α and Emission Lines

In [ ]:
from tangelo import spectroscopy as tgspec
from tangelo import plotting as tgplot
from tangelo import constants as tgconst
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import astropy.table as aptb
from matplotlib import gridspec

In [ ]:
# Load megatable of Lyman alpha sources
# Which spectra source to use? R21 or APER
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = 'weight_skysub' if SPEC_SOURCE == "R21" else '2fwhm'

tabfile = f"./megatables/lae_megatab_flagged_cpts_allrefit_zeldamcmc_sysz_absv_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

# Filter to only include LAEs
lya_mask = (megatable['SNRR'] >= 10) | (megatable['SNRB'] >= 10)
megatable = megatable[lya_mask]

In [ ]:
# Find out how many absorbers there are in total by looking for rows that have
# either DV_LI_ABS, DV_HI_ABS, or DV_TOT_ABS not NaN
num_absorbers = np.sum(~np.isnan(megatable['DV_TOT_ABS']) 
                       + ~np.isnan(megatable['DV_LI_ABS'])
                       + ~np.isnan(megatable['DV_HI_ABS']))

print(f"Total number of absorbers: {num_absorbers}")

# How many of these also have systemic velocity measurements?
has_sysvel = ~np.isnan(megatable['DELTAV_LYA'])
has_absorber = (~np.isnan(megatable['DV_TOT_ABS']) 
                + ~np.isnan(megatable['DV_LI_ABS'])
                + ~np.isnan(megatable['DV_HI_ABS'])) > 0

num_absorbers_with_sysvel = np.sum(has_sysvel & has_absorber)
print(f"Number of absorbers with systemic velocity measurements: {num_absorbers_with_sysvel}")

In [ ]:
# Generate a figure containing all absorbers with systemic velocity measurements

# These are the lines that, if detected at >3 sigma, can be used to make the stack
good_lines =  ['CIII1907', 'CIII1909', 'HeII1640', 
               'NIII1750', 'NIV1483', 'NIV1487',
               'OIII1660', 'OIII1666', 'SiIII1883', 'SiIII1892']

# These are the default optically thin lines to use if no good lines are detected
optthin_lines = ['HeII1640','OIII1666','CIII1907','CIII1909']

# First figure out dimensions: name how many columns we want, and calculate rows
num_cols = 4
num_rows = int(np.ceil(num_absorbers_with_sysvel / num_cols))
# How many sub-subplots for each subplot?
num_subs = 4 # Lyman alpha, emission lines, low-ion absorption, high-ion absorption

fig = plt.figure(figsize=(5*num_cols,5*num_rows), facecolor='white')
gs  = gridspec.GridSpec(num_rows, num_cols, wspace=0.1, hspace=0.15, figure=fig)

for i, row in enumerate(megatable[has_sysvel & has_absorber]):
    # Determine which subplot we're in
    row_idx = i // num_cols
    col_idx = i % num_cols
    sub_gs = gridspec.GridSpecFromSubplotSpec(4, 1, subplot_spec=gs[row_idx, col_idx], wspace=0, hspace=0)

    # Plot Lyman alpha (shifted to systemic)
    ax0 = fig.add_subplot(sub_gs[0])
    lya_wl, lya_spec, lya_err = tgspec.avg_lines(row, ['LYALPHA'], velbounds=[-2500, 2500])
    lya_wl_shifted = lya_wl + row['DELTAV_LYA']  # Shift to systemic velocity

    ax0.plot(lya_wl_shifted, lya_spec, color='slateblue', drawstyle='steps-mid', label=r"Ly$\alpha$")
    ax0.fill_between(lya_wl_shifted, lya_spec - lya_err, lya_spec + lya_err, color='slateblue', alpha=0.3,
                     step='mid', linestyle='')
    ax0.axvline(row['DELTAV_LYA'], color='red', linestyle='--', alpha=0.7)
    ax0.tick_params(labelbottom=False)  # Hide x-tick labels on top row
    ax0.set_xlim(-2100, 2100)
    # Only put the legend on the first subplot
    if i == 0:
        ax0.legend(frameon=True, facecolor='white', edgecolor='none', framealpha=0.8)

    # Calculate the true redshift using Lyman alpha peak redshift and velocity offset from systemic
    c_kms = tgconst.c
    true_z = ((row['LPEAKR'] / tgconst.wavedict['LYALPHA']) * (1 - row['DELTAV_LYA'] / c_kms)) - 1
    
    # Add object ID text in upper left corner
    ax0.text(0.02, 0.90, f"{row['CLUSTER']} {row['iden']}\n($z={true_z:.3f}$)", 
             transform=ax0.transAxes, fontsize=12, fontweight='bold',
             verticalalignment='top', horizontalalignment='left',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none'))

    # Plot stacked emission lines for systemic velocity (shifted to systemic)
    # Which emission lines to use?
    # First check which of the good lines were detected at >3 sigma
    em_lines = [x if row[f'SNR_{x}'] >= 3.0 else None for x in good_lines]
    em_lines = [x for x in em_lines if x is not None]
    if len(em_lines) == 0:
        print('No optically thin lines detected, reverting to default line list...')
        em_lines = optthin_lines

    print(f"Using emission lines for object ID {row['iden']}: {em_lines}")

    ax1 = fig.add_subplot(sub_gs[1], sharex=ax0)
    emis_wl, emis_spec, emis_err = tgspec.avg_lines(row, em_lines, velbounds=[-2500, 2500])
    emis_wl_shifted = emis_wl + row['DELTAV_LYA']  # Shift to systemic velocity

    ax1.plot(emis_wl_shifted, emis_spec, color='firebrick', drawstyle='steps-mid', label='Opt. thin emission')
    ax1.fill_between(emis_wl_shifted, emis_spec - emis_err, emis_spec + emis_err, color='firebrick', alpha=0.3,
                      step='mid', linestyle='')
    ax1.axvline(0., color='red', linestyle='--', alpha=0.7)
    ax1.tick_params(labelbottom=False)  # Hide x-tick labels on second row
    # Only put the legend and axis label on the first subplot
    if i == 0:
        ax1.set_ylabel('$f_v$ [erg\,s$^{-1}$\,cm$^{-2}$\,(km\,s$^{-1}$)$^{-1}$]')
        ax1.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none', framealpha=0.8)

    # Determine which line combinations were used to detect absorption
    # First, collect what was actually detected
    raw_detected = {}
    for line_type in ['LI', 'HI', 'TOT']:
        if line_type == 'HI' and not np.isnan(row['DV_HI_ABS']):
            raw_detected['HI'] = "SiIV1394+SiIV1403"
            print(f"Object ID {row['iden']} has HI absorption from SiIV1394 and SiIV1403")
        elif line_type in ['LI', 'TOT'] and not np.isnan(row[f'DV_{line_type}_ABS']):
            dv_lines_key = f'DV_{line_type}_ABS_LINES'
            raw_detected[line_type] = row[dv_lines_key]
            print(f"Object ID {row['iden']} has {line_type} absorption from the following lines: {row[dv_lines_key]}")

    # Now build the detected_types dictionary with proper ordering
    # Rule: Always start with LI
    # If LI or HI detected -> plot LI, HI
    # If only TOT detected -> plot LI, TOT
    detected_types = {}
    
    if 'LI' in raw_detected or 'HI' in raw_detected:
        # LI or HI detected: plot LI first, then HI
        if 'LI' in raw_detected:
            detected_types['LI'] = raw_detected['LI']
        else:
            detected_types['LI'] = 'SiII1260+CII1334'  # Generic LI
        
        if 'HI' in raw_detected:
            detected_types['HI'] = raw_detected['HI']
        else:
            detected_types['HI'] = 'SiIV1394+SiIV1403'  # Generic HI
    
    elif 'TOT' in raw_detected:
        # Only TOT detected: plot LI first, then TOT
        detected_types['LI'] = 'SiII1260+CII1334'  # Generic LI
        detected_types['TOT'] = raw_detected['TOT']

    # Generate axes for absorption lines
    absaxes = {type: fig.add_subplot(sub_gs[2 + idx], sharex=ax0) for idx, type in enumerate(detected_types.keys())}
    typecolors = {'LI': 'teal', 'HI': 'darkgoldenrod', 'TOT': 'indigo'}

    for idx, (abstype, lines) in enumerate(detected_types.items()): # Loop through the detected absorption types, average the lines used and plot

        # Generate list of lines from string
        line_list = lines.split('+')

        # Load the spectrum for this object
        spectab    = tgspec.avg_lines(row, line_list, velbounds = [-2500, 2500])
        wavelength = spectab[0]
        flux       = spectab[1]
        flux_err   = spectab[2]
        
        # Shift wavelength to systemic velocity
        wavelength_shifted = wavelength + row['DELTAV_LYA']

        # Plot the absorption
        ax = absaxes[abstype]
        plot_label = '\n'.join(line_list)
        ax.plot(wavelength_shifted, flux, color=typecolors[abstype], label=plot_label if abstype !='TOT' else 'total absorption',
                 drawstyle='steps-mid')
        ax.fill_between(wavelength_shifted, flux - flux_err, flux + flux_err, color=typecolors[abstype], 
                        alpha=0.3, step='mid', linestyle='')
        dv_abs = row[f'DV_{abstype}_ABS']
        dv_abs_shifted = dv_abs + row['DELTAV_LYA']  # Shift absorber velocity to systemic
        ax.axvline(dv_abs_shifted, color='red', linestyle='--', alpha=0.7)
        
        # Only show x-tick labels on the last (bottom) absorption subplot
        is_last = (idx == len(detected_types) - 1)
        if is_last:
            ax.set_xlabel('Velocity relative to systemic (km/s)')
            # Explicitly enable x-tick labels
            ax.tick_params(labelbottom=True)
        else:
            ax.tick_params(labelbottom=False)

        ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='none', framealpha=0.8)

# plt.suptitle("Lyman Alpha Sources with Absorbers and Systemic Velocities", fontsize=16)
plt.savefig(f"/home/james/Documents/my_papers/lensed_laes/paper2/figures/absorption_profiles_{SPEC_TYPE}.pdf", 
            dpi=500, bbox_inches='tight')
plt.show()
plt.close(fig)
